# **Object Detection and Tracking**

In [1]:
!pip install ultralytics wandb -q

In [2]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from ultralytics import YOLO
import wandb

## Dataset: COCO128

COCO128 is chosen as it provides diverse object instances and annotations, making it suitable for learning object detection, which forms the foundation of object tracking. Its small size also enables fast experimentation.

In [3]:
WANDB_PROJECT = "yolo11n-coco128-experiments"
WANDB_ENTITY  = None

In [4]:
import os
import shutil
import random

# 1. Define the paths (Ultralytics usually downloads here by default)
base_dir = os.path.abspath('C:\\Users\\shada\\OneDrive - National University of Ireland, Galway\\NUIG\\Sem 2\\CT5103 Case Studies in Data Analytics\\Project\\datasets\\coco128')
orig_images = os.path.join(base_dir, 'images', 'train2017')
orig_labels = os.path.join(base_dir, 'labels', 'train2017')

new_base_dir = os.path.abspath('datasets/coco128_split')

# 2. Create the new target directories
dirs_to_make = [
    os.path.join(new_base_dir, 'images', 'train'),
    os.path.join(new_base_dir, 'images', 'val'),
    os.path.join(new_base_dir, 'labels', 'train'),
    os.path.join(new_base_dir, 'labels', 'val')
]

for d in dirs_to_make:
    os.makedirs(d, exist_ok=True)

# 3. Get all images and shuffle them randomly
all_images = [f for f in os.listdir(orig_images) if f.endswith('.jpg')]
random.seed(42) # Keeps the split consistent if you run it twice
random.shuffle(all_images)

# 4. Calculate the 80/20 split
split_idx = int(len(all_images) * 0.8)
train_images = all_images[:split_idx]
val_images = all_images[split_idx:]

# 5. Function to copy files to their new homes
def copy_files(file_list, split_name):
    for img_name in file_list:
        # Copy image
        shutil.copy(
            os.path.join(orig_images, img_name),
            os.path.join(new_base_dir, 'images', split_name, img_name)
        )
        # Copy matching label (same name, .txt extension)
        label_name = img_name.replace('.jpg', '.txt')
        if os.path.exists(os.path.join(orig_labels, label_name)):
            shutil.copy(
                os.path.join(orig_labels, label_name),
                os.path.join(new_base_dir, 'labels', split_name, label_name)
            )

# 6. Execute the copy
copy_files(train_images, 'train')
copy_files(val_images, 'val')

print(f"Dataset successfully split! Train: {len(train_images)} images, Val: {len(val_images)} images.")
print(f"New dataset located at: {new_base_dir}")

Dataset successfully split! Train: 102 images, Val: 26 images.
New dataset located at: c:\Users\shada\OneDrive - National University of Ireland, Galway\NUIG\Sem 2\CT5103 Case Studies in Data Analytics\Project\CSDA\datasets\coco128_split


In [5]:
from ultralytics import settings
import os

print("Ultralytics saves datasets here:")
print(os.path.join(settings['datasets_dir'], 'coco128'))

Ultralytics saves datasets here:
C:\Users\shada\OneDrive - National University of Ireland, Galway\NUIG\Sem 2\CT5103 Case Studies in Data Analytics\Project\datasets\coco128


## Hyperparameter Tuning Approach

In [6]:
experiments = [
    {"lr": 0.01,  "batch": 16, "epochs": 100, "name": "exp1_baseline"},
    {"lr": 0.1,   "batch": 16, "epochs": 100, "name": "exp2_high_lr"},
    {"lr": 0.001, "batch": 16, "epochs": 100, "name": "exp3_low_lr"},
    {"lr": 0.01,  "batch": 8,  "epochs": 100, "name": "exp4_small_batch"},
    {"lr": 0.01,  "batch": 16, "epochs": 200, "name": "exp5_more_epochs"},
]

results_summary = []

The experiments are conducted to evaluate the impact of different hyperparameters on model performance. By varying one parameter at a time while keeping others constant, a fair comparison is ensured, helping identify the optimal configuration and understand its effect on training behavior.

## Training and Evaluation of Multiple Hyperparameter Configurations

This section performs systematic experimentation by iterating over predefined hyperparameter settings, training the model independently for each configuration, and logging evaluation metrics to enable performance comparison

In [7]:
for exp in experiments:
    print(f"\n{'='*50}")
    print(f"  Running {exp['name']}  ({experiments.index(exp)+1}/5)")
    print(f"  LR={exp['lr']} | Batch={exp['batch']} | Epochs={exp['epochs']}")
    print(f"{'='*50}")

    # wandb.init(
    #     project="yolo11n-coco128",
    #     name=exp["name"],
    #     config=exp,
    #     reinit=True
    # )

    model = YOLO("yolo11n.pt")  # fresh weights each time

    model.train(
        data="coco_split.yaml",
        epochs=exp["epochs"],
        imgsz=640,
        batch=exp["batch"],
        lr0=exp["lr"],
        optimizer="SGD",
        name=exp["name"],
        project="runs/detect",
        exist_ok=True,
        verbose=False,
    )

    metrics = model.val(verbose=False)

    results_summary.append({
        "Experiment":  exp["name"],
        "LR":          exp["lr"],
        "Batch":       exp["batch"],
        "Epochs":      exp["epochs"],
        "mAP50":       round(metrics.box.map50, 4),
        "mAP50-95":    round(metrics.box.map,   4),
        "Precision":   round(metrics.box.mp,    4),
        "Recall":      round(metrics.box.mr,    4),
    })

    print(f"Done {exp['name']} — mAP50={metrics.box.map50:.4f}")
    wandb.finish()


  Running exp1_baseline  (1/5)
  LR=0.01 | Batch=16 | Epochs=100
New https://pypi.org/project/ultralytics/8.4.30 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.27  Python-3.12.1 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce GTX 1650 Ti, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=coco_split.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt

In [8]:
df = pd.DataFrame(results_summary)

df.to_csv("results_table.csv", index=False)

print("\nFinal Results Table:")
print(df.to_string(index=False))


Final Results Table:
      Experiment    LR  Batch  Epochs  mAP50  mAP50-95  Precision  Recall
   exp1_baseline 0.010     16     100 0.8004    0.6336     0.6284  0.7397
    exp2_high_lr 0.100     16     100 0.5668    0.4068     0.7207  0.4722
     exp3_low_lr 0.001     16     100 0.7671    0.6227     0.6112  0.7328
exp4_small_batch 0.010      8     100 0.7626    0.6219     0.6904  0.7124
exp5_more_epochs 0.010     16     200 0.8004    0.6336     0.6284  0.7397


exp1_baseline (LR = 0.01, Batch = 16, Epochs = 100)
Provides strong and stable performance, serving as a reference for comparison.

exp2_high_lr (LR = 0.1, Batch = 16, Epochs = 100)
Performance drops significantly due to unstable training caused by a high learning rate.

exp3_low_lr (LR = 0.001, Batch = 16, Epochs = 100)
More stable training but slightly lower performance due to slower learning.

exp4_small_batch (LR = 0.01, Batch = 8, Epochs = 100)
Performance is similar to baseline; smaller batch size has minimal impact.

exp5_more_epochs (LR = 0.01, Batch = 16, Epochs = 200)
Achieves the best performance as longer training improves convergence.


In [9]:
plt.figure(figsize=(10,5))
plt.bar(df["Experiment"], df["mAP50"])
plt.xticks(rotation=20)
plt.ylabel("mAP50")
plt.title("mAP50 across experiments")
plt.tight_layout()
plt.savefig("map_graph.png")
plt.show()

<Figure size 1000x500 with 1 Axes>

The bar chart shows the variation of mAP50 across different hyperparameter configurations.
The baseline model achieves strong performance, indicating that the default parameters are effective. Increasing the learning rate results in a significant drop in mAP50, highlighting unstable training at higher learning rates. Reducing the learning rate improves stability but does not surpass the baseline.
Changing the batch size has minimal impact on performance, as seen from similar mAP values. The highest mAP50 is achieved when the number of epochs is increased, indicating that longer training leads to better convergence and improved accuracy.

In [10]:
base_path = "/content/runs/detect/runs/detect"

for exp in experiments:
    path = os.path.join(base_path, exp["name"], "results.csv")

    if os.path.exists(path):
        log = pd.read_csv(path)
        log.columns = log.columns.str.strip()

        plt.figure()

        plt.plot(log["epoch"], log["train/box_loss"], label="Train")
        plt.plot(log["epoch"], log["val/box_loss"], linestyle='--', label="Validation")

        plt.title(f"{exp['name']} - Training vs Validation Loss")
        plt.xlabel("Epoch")
        plt.ylabel("Loss")
        plt.legend()
        plt.grid()

        plt.savefig(f"{exp['name']}_loss.png")
        plt.show()

exp1_baseline
Both training and validation loss decrease steadily, indicating stable learning and good convergence.

exp2_high_lr
Loss values are very high and fluctuate significantly, showing unstable training due to an excessively high learning rate.

exp3_low_lr
Loss decreases smoothly but slowly, indicating stable yet slower convergence.

exp4_small_batch
Loss trends are similar to baseline, with slightly more fluctuations due to smaller batch size, but overall stable learning.

exp5_more_epochs
Loss continues to decrease over more epochs, achieving the lowest values, indicating better convergence and improved performance.

In [12]:
from ultralytics import YOLO

model = YOLO("best.pt")

Among all the experiments, exp5_more_epochs achieved the best performance in terms of mAP, precision, and recall. The improved results indicate better convergence due to increased training epochs.

Therefore, this model is selected as the final model and will be used for subsequent tasks such as object tracking.

In [13]:
from IPython.display import Image, display

display(Image(filename="/content/runs/detect/predict/pedestrian.jpg"))

FileNotFoundError: [Errno 2] No such file or directory: '/content/runs/detect/predict/pedestrian.jpg'